In [ ]:
# =====================================================================
# 1. IMPORTS WITH DESCRIPTIVE GROUPING
# =====================================================================
import os
import sys
import copy
import math
import random
import time
import warnings
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warning outputs to maintain clean terminal execution logs
warnings.filterwarnings('ignore')


# loading data



# =====================================================================
# 2. SEED SELECTION AND SYSTEM HYPERPARAMETERS
# =====================================================================
SEED = 42
DATA_PATH = "/content/gdrive/MyDrive/data.csv" # Updated path from /content/drive to /content/gdrive
TARGET_COLUMN = "is_attack"
SELECTED_CLASSES = [0, 1]

# Ensemble architecture definitions
NUM_TEACHERS = 9
TEACHER_ARCHITECTURES = ['LSTM', 'LSTM', 'LSTM', 'GRU', 'GRU', 'GRU', 'CNN', 'CNN', 'CNN']
STUDENT_ARCHITECTURE = "MLP"

# Training schedules and optimizations
BATCH_SIZE = 128
TEACHER_EPOCHS = 23
STUDENT_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE = 3

# Stratified split partitions mapping to 100% of data
TEACHER_VALID_FRACTION = 0.30
STUDENT_VALID_FRACTION = 0.20
TEACHER_POOL_FRACTION = 0.80
STUDENT_PRIVATE_FRACTION = 0.00
PUBLIC_FRACTION = 0.05
FINAL_TEST_FRACTION = 0.15

# Laplace Sanitization and DP moments accountant parameters
LAPLACE_INVERSE_SCALE = 1.3
DELTA = 1e-5
MAX_MOMENT = 32
USE_OVERSAMPLING = True
OUTPUT_DIR = "./pate_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Automatically route calculations to CUDA if an active GPU is present
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 3. DATASET LOADING AND PIPELINE SPLITTING
# =====================================================================
def load_dataset(path, target_column, selected_classes):
    """
    Loads a target CSV file and filters the raw data to extract only observations
    corresponding to a specified list of network threat classes.
    """
    data = pd.read_csv(path, index_col=0)
    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' was not found.")
    data = data[data[target_column].isin(selected_classes)].copy()
    if len(data) == 0:
        raise ValueError("No observations remain after class filtering.")
    return data

def preprocess_and_split(data, target_column, seed=42):
    """
    Performs numeric conversions, sanitizes missing records, encodes multi-class labels,
    and executes stratified dataset partitioning to construct private, public, and test splits.
    """
    data = data.copy()
    total_fraction = TEACHER_POOL_FRACTION + STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    if not np.isclose(total_fraction, 1.0):
        raise ValueError("Dataset fractions must sum to 1.0.")

    # Encode target labels numerically
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(data[target_column])
    X = data.drop(columns=[target_column])

    # Exclude non-numerical feature descriptors
    non_numeric = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(non_numeric) > 0:
        X = X.drop(columns=non_numeric)

    # Enforce strict float datatypes and discard NaN records
    X = X.apply(pd.to_numeric, errors="coerce")
    valid_rows = ~X.isna().any(axis=1)
    X = X.loc[valid_rows].reset_index(drop=True)
    y = y[valid_rows.to_numpy()]

    X = X.to_numpy(dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    # Separate raw teacher pool from evaluation splits
    X_teacher, X_heldout, y_teacher, y_heldout = train_test_split(
        X, y, test_size=(1.0 - TEACHER_POOL_FRACTION), random_state=seed, stratify=y
    )

    # Sub-partition heldout records into Public/Unlabeled (Student Training) and Final Test structures
    heldout_total_sub_fraction = STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    test_size_final_test = FINAL_TEST_FRACTION / heldout_total_sub_fraction

    X_public, X_final_test, y_public, y_final_test = train_test_split(
        X_heldout, y_heldout, test_size=test_size_final_test, random_state=seed, stratify=y_heldout
    )

    # Initialize empty arrays for student private data (set to 0.0 fraction)
    X_private = np.zeros((0, X_teacher.shape[1]), dtype=np.float32)
    y_private = np.zeros((0,), dtype=np.int64)

    return X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder

# =====================================================================
# 4. PYTORCH UTILITY DATASET IMPLEMENTATIONS
# =====================================================================
class TabularDataset(Dataset):
    """
    A minimal Dataset wrapper to facilitate mini-batch delivery of feature
    and label vectors through standard PyTorch DataLoaders.
    """
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

def make_loader(X, y, batch_size, shuffle):
    """
    Constructs a simple, optimized DataLoader wrapper around the TabularDataset.
    """
    return DataLoader(TabularDataset(X, y), batch_size=batch_size, shuffle=shuffle, num_workers=0)

# =====================================================================
# 5. CORE ENSEMBLE TEACHER ARCHITECTURES
# =====================================================================
class TeacherLSTM(nn.Module):
    """
    Recurrent architecture using an LSTM layer for temporal patterns in sequences.
    """
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherGRU(nn.Module):
    """
    Gated Recurrent Unit architecture to process inputs sequentially.
    """
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherCNN(nn.Module):
    """
    1D CNN architecture mapping localized, neighboring feature representations.
    """
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class TeacherMLP(nn.Module):
    """
    Fully-connected Multi-Layer Perceptron architecture with regularizing dropout.
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

class StudentMLP(nn.Module):
    """
    Standard fully connected Student Multi-Layer Perceptron for PATE distillation.
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

# =====================================================================
# 6. MODEL FACTORY FUNCTIONS
# =====================================================================
def build_teacher(architecture, input_size, num_classes):
    """
    Resolves and instantiates the proper teacher model architecture based on input string.
    """
    if architecture == "LSTM":
         return TeacherLSTM(input_size=input_size, num_classes=num_classes)
    if architecture == "GRU":
         return TeacherGRU(input_size=input_size, num_classes=num_classes)
    if architecture == "CNN":
        return TeacherCNN(input_size=input_size, num_classes=num_classes)
    if architecture == "MLP":
        return TeacherMLP(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown teacher architecture: {architecture}")

# =====================================================================
# 7. TRAINING, EVALUATION AND PREDICTION FUNCTIONS
# =====================================================================
def train_model(model, train_loader, valid_loader, epochs, learning_rate, patience, device):
    """
    Trains a PyTorch neural network model using cross-entropy, optimization,
    evaluation on a validation loader, and early stopping checkpoint recovery.
    """
    model = model.to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_validation_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        total_train_loss, total_train, correct_train = 0.0, 0, 0
        for is_input, labels in train_loader:
            is_input, labels = is_input.to(device), labels.to(device)
            optimizer.zero_grad()
            log_probabilities = model(is_input)
            loss = criterion(log_probabilities, labels)
            loss.backward()
            optimizer.step()
            b_size = labels.size(0)
            total_train_loss += loss.item() * b_size
            predictions = torch.argmax(log_probabilities, dim=1)
            correct_train += (predictions == labels).sum().item()
            total_train += b_size

        model.eval()
        total_valid_loss, total_valid, correct_valid = 0.0, 0, 0
        with torch.no_grad():
            for is_input, labels in valid_loader:
                is_input, labels = is_input.to(device), labels.to(device)
                log_probabilities = model(is_input)
                loss = criterion(log_probabilities, labels)
                b_size = labels.size(0)
                total_valid_loss += loss.item() * b_size
                predictions = torch.argmax(log_probabilities, dim=1)
                correct_valid += (predictions == labels).sum().item()
                total_valid += b_size

        valid_loss = total_valid_loss / total_valid
        history.append({
            "epoch": epoch + 1,
            "train_loss": total_train_loss / total_train,
            "valid_loss": valid_loss,
            "train_accuracy": correct_train / total_train,
            "valid_accuracy": correct_valid / total_valid
        })

        # Process Early Stopping criteria
        if valid_loss < best_validation_loss:
            best_validation_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    # Restore model to its optimal parameter weights validation epoch state
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

def predict_labels(model, loader, device):
    """
    Inferences predictions for unlabeled instances in loader using forward pass.
    """
    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = model(inputs)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            predictions.extend(batch_predictions.cpu().numpy())
    return np.asarray(predictions, dtype=np.int64)

def evaluate_model(model, X_test, y_test, batch_size, device):
    """
    Evaluates accuracy, macro precision, recall, and F1 macro on test splits.
    """
    model.eval()
    loader = DataLoader(TabularDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    all_predictions = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = model(inputs)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            all_predictions.extend(batch_predictions.cpu().numpy())

    all_predictions = np.array(all_predictions)
    accuracy = accuracy_score(y_test, all_predictions)
    precision = precision_score(y_test, all_predictions, average='macro', zero_division=0)
    recall = recall_score(y_test, all_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, all_predictions, average='macro', zero_division=0)
    cm = confusion_matrix(y_test, all_predictions)

    metrics = {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1_macro
    }
    return metrics, cm, all_predictions

# =====================================================================
# 8. PATE VOTE AGGREGATION UTILITIES
# =====================================================================
def teacher_predictions_to_counts(teacher_preds, num_classes):
    """
    Converts flat multi-teacher predictions arrays into structured sample-wise class bins.
    """
    teacher_preds = np.asarray(teacher_preds, dtype=np.int64)
    num_teachers, num_queries = teacher_preds.shape
    counts = np.zeros((num_queries, num_classes), dtype=np.int64)
    for query_index in range(num_queries):
        counts[query_index] = np.bincount(teacher_preds[:, query_index], minlength=num_classes)
    return counts

def aggregate_teacher_votes_laplace(teacher_preds, num_classes, laplace_inverse_scale, random_seed=None):
    """
    Adds Laplace noise to vote counts of each label to yield private consensus targets.
    """
    counts = teacher_predictions_to_counts(teacher_preds, num_classes)
    rng = np.random.default_rng(random_seed)
    laplace_scale = 1.0 / laplace_inverse_scale
    noise = rng.laplace(loc=0.0, scale=laplace_scale, size=counts.shape)
    noisy_counts = counts.astype(np.float64) + noise
    released_labels = np.argmax(noisy_counts, axis=1)
    return released_labels, counts, noisy_counts

# =====================================================================
# 9. DIFFERENTIAL PRIVACY ACCOUNTANT (RÉNYI DIFFERENTIAL PRIVACY)
# =====================================================================
def logaddexp_scalar(a, b):
    """
    Calculates log(exp(a) + exp(b)) safely avoiding numerical underflow.
    """
    if a == -math.inf: return b
    if b == -math.inf: return a
    maximum = max(a, b)
    return maximum + math.log(math.exp(a - maximum) + math.exp(b - maximum))

def logmgf_exact(q, mechanism_eps, moment_order):
    """
    Rényi Differential Privacy (RDP) log-MGF bounds derived analytically.
    """
    lam = float(moment_order)
    eps = float(mechanism_eps)
    bound_1 = 0.5 * (eps ** 2) * lam * (lam + 1.0)
    bound_3 = eps * lam
    bound_2 = math.inf
    denominator = 1.0 - math.exp(eps) * q
    if q < 0.5 and denominator > 0.0:
        log_one_minus_q = math.log1p(-q)
        log_term_1 = log_one_minus_q + lam * (log_one_minus_q - math.log(denominator))
        log_term_2 = math.log(q) + eps * lam if q > 0.0 else -math.inf
        bound_2 = logaddexp_scalar(log_term_1, log_term_2)
    return min(bound_1, bound_2, bound_3)

def compute_q_noisy_max(counts, laplace_inverse_scale):
    """
    Estimates bound on maximum probability q of an alternative label selection.
    """
    counts = np.asarray(counts, dtype=np.float64)
    num_classes = len(counts)
    if num_classes < 2:
        return 0.0
    winner = int(np.argmax(counts))
    winning_votes = counts[winner]
    competitors = np.delete(counts, winner)
    gaps = laplace_inverse_scale * (winning_votes - competitors)
    q = float(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0))
    return min(max(q, 0.0), 1.0 - 1.0 / num_classes)

def data_dependent_log_moment(counts, laplace_inverse_scale, moment_order):
    """
    Computes local data dependent log-moment for private queries.
    """
    q = compute_q_noisy_max(counts, laplace_inverse_scale)
    mechanism_eps = 2.0 * laplace_inverse_scale
    return logmgf_exact(q, mechanism_eps, moment_order)

def data_independent_log_moment(laplace_inverse_scale, moment_order):
    """
    Computes worst-case theoretical data independent log-moment.
    """
    mechanism_eps = 2.0 * laplace_inverse_scale
    lam = float(moment_order)
    concentration_bound = 0.5 * (mechanism_eps ** 2) * lam * (lam + 1.0)
    direct_dp_bound = mechanism_eps * lam
    return min(concentration_bound, direct_dp_bound)

def calculate_pate_privacy(vote_counts, laplace_inverse_scale=1.3, delta=1e-5, moments=32, expected_num_teachers=None):
    """
    Computes total data-dependent and independent epsilon bounds from query vote profiles.
    """
    vote_counts = np.asarray(vote_counts, dtype=np.int64)
    num_queries, classes_dim = vote_counts.shape
    orders = np.arange(1, moments + 1, dtype=np.float64)
    dependent_total = np.zeros(moments, dtype=np.float64)
    independent_total = np.zeros(moments, dtype=np.float64)
    q_values, vote_margins = [], []

    for counts in vote_counts:
        q = compute_q_noisy_max(counts, laplace_inverse_scale)
        q_values.append(q)
        sorted_counts = np.sort(counts)
        vote_margins.append(sorted_counts[-1] - sorted_counts[-2])
        for pos, order in enumerate(orders):
            dependent_total[pos] += data_dependent_log_moment(counts, laplace_inverse_scale, order)
            independent_total[pos] += data_independent_log_moment(laplace_inverse_scale, order)

    dep_candidates = (dependent_total - math.log(delta)) / orders
    ind_candidates = (independent_total - math.log(delta)) / orders
    dep_idx = np.argmin(dep_candidates)
    ind_idx = np.argmin(ind_candidates)

    return {
        "data_dependent_epsilon": float(dep_candidates[dep_idx]),
        "data_independent_epsilon": float(ind_candidates[ind_idx]),
        "q_values": np.asarray(q_values),
        "vote_margins": np.asarray(vote_margins)
    }

# =====================================================================
# 10. COMPLIANCE UNIT TESTS
# =====================================================================
def run_accounting_tests():
    """
    Verifies mathematical correctness of accountant outputs using established
    closed-form analytical solutions.
    """
    print("\n" + "=" * 70)
    print("RUNNING PATE ACCOUNTING COMPLIANCE TESTS")
    print("=" * 70)
    counts = np.array([10, 6, 2], dtype=float)
    eta = 0.1
    gaps = eta * np.array([4.0, 8.0])
    expected_q = min(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0), 2.0 / 3.0)
    actual_q = compute_q_noisy_max(counts, eta)
    assert np.isclose(actual_q, expected_q), f"q discrepancy: {actual_q} vs {expected_q}"
    print("PASS: Direct multiclass q verification")

    di_test_val = data_independent_log_moment(laplace_inverse_scale=1.0, moment_order=2)
    assert np.isclose(di_test_val, 4.0), f"DI log moment discrepancy: {di_test_val} vs 4.0"
    print("PASS: Independent hand-calculated DI regression test")
    print("All core compliance unit tests passed successfully!")

# =====================================================================
# 11. BASE SINGLE EXPERIMENT RUNNER
# =====================================================================
def run_full_pate_experiment(config):
    """
    Orchestrates training and privacy verification for baseline evaluation setups.
    """
    print(f"\nExecuting experiment run: {config['description']}")

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    data = load_dataset(DATA_PATH, TARGET_COLUMN, SELECTED_CLASSES)
    X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder = preprocess_and_split(data, TARGET_COLUMN, seed=SEED)

    input_size = X_teacher.shape[1]
    num_classes = len(np.unique(y_teacher))

    scaler = StandardScaler()
    X_teacher_scaled = scaler.fit_transform(X_teacher)
    X_public_scaled = scaler.transform(X_public)
    X_final_test_scaled = scaler.transform(X_final_test)

    n = len(X_teacher_scaled)
    indices = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    partitions = np.array_split(indices, config["num_teachers"])

    train_loaders, valid_loaders = [], []
    for teacher_id, teacher_indices in enumerate(partitions):
        X_part = X_teacher_scaled[teacher_indices]
        y_part = y_teacher[teacher_indices]
        X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + teacher_id, stratify=y_part)
        if USE_OVERSAMPLING:
            sampler = RandomOverSampler(random_state=SEED + teacher_id)
            X_train, y_train = sampler.fit_resample(X_train, y_train)
        train_loaders.append(DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True))
        valid_loaders.append(DataLoader(TabularDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False))

    teachers = []
    for idx in range(config["num_teachers"]):
        arch = config["teacher_archs"][idx % len(config["teacher_archs"])]
        model = build_teacher(arch, input_size, num_classes)
        trained_model, _ = train_model(model, train_loaders[idx], valid_loaders[idx], TEACHER_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)
        teachers.append(trained_model)

    public_loader = DataLoader(TabularDataset(X_public_scaled, y_public), batch_size=BATCH_SIZE, shuffle=False)
    all_teacher_preds = []
    for idx, t_model in enumerate(teachers):
        preds = predict_labels(t_model, public_loader, DEVICE)
        all_teacher_preds.append(preds)
    all_teacher_preds = np.stack(all_teacher_preds)

    released_labels, vote_counts, _ = aggregate_teacher_votes_laplace(all_teacher_preds, num_classes, LAPLACE_INVERSE_SCALE, random_seed=SEED)
    privacy_results = calculate_pate_privacy(vote_counts, LAPLACE_INVERSE_SCALE, DELTA, MAX_MOMENT)

    X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED, stratify=released_labels)
    student_train_loader = DataLoader(TabularDataset(X_student_train, y_student_train), batch_size=BATCH_SIZE, shuffle=True)
    student_valid_loader = DataLoader(TabularDataset(X_student_valid, y_student_valid), batch_size=BATCH_SIZE, shuffle=False)

    student_model = StudentMLP(input_size=input_size, num_classes=num_classes)
    trained_student, _ = train_model(student_model, student_train_loader, student_valid_loader, STUDENT_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)

    final_metrics, cm, _ = evaluate_model(trained_student, X_final_test_scaled, y_final_test, BATCH_SIZE, DEVICE)

    code_str = inspect.getsource(calculate_pate_privacy) + inspect.getsource(compute_q_noisy_max)
    code_hash = hashlib.sha256(code_str.encode('utf-8')).hexdigest()

    audit_log = {
        "configuration": config["description"],
        "num_teachers": config["num_teachers"],
        "epsilon_data_dependent": privacy_results["data_dependent_epsilon"],
        "epsilon_data_independent": privacy_results["data_independent_epsilon"],
        "student_test_accuracy": final_metrics["accuracy"],
        "student_test_f1_macro": final_metrics["f1_macro"],
        "code_version_hash": code_hash
    }

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sanitized_desc = config["description"].lower().replace(" ", "_").replace(":", "").replace("(", "").replace(")", "").replace(",", "")
    audit_record_path = os.path.join(OUTPUT_DIR, f"audit_record_{sanitized_desc}.json")
    with open(audit_record_path, "w") as f:
        json.dump(audit_log, f, indent=4)

    return {
        "teachers": teachers,
        "student": trained_student,
        "privacy_results": privacy_results,
        "final_metrics": final_metrics,
        "confusion_matrix": cm,
        "audit_record_path": audit_record_path
    }

# =====================================================================
# 12. OPTIONAL VISUALIZATION ROUTINES
# =====================================================================
def plot_privacy_loss_composition(experiment_results):
    """
    Outputs calculated Epsilon vectors onto console terminal.
    """
    priv_res = experiment_results['privacy_results']
    print(f"Data-Dependent Epsilon: {priv_res['data_dependent_epsilon']:.6f}")
    print(f"Data-Independent Epsilon: {priv_res['data_independent_epsilon']:.6f}")

# =====================================================================
# 13. ENTRYPOINT HANDLER
# =====================================================================
def main():
    print("Starting consolidated execution block...")
    run_accounting_tests()

    main_cfg = {
        'description': 'PATE: 9 Teachers (3 LSTM, 3 GRU, 3 CNN), 1 MLP Student',
        'num_teachers': NUM_TEACHERS,
        'teacher_archs': TEACHER_ARCHITECTURES,
        'student_arch': STUDENT_ARCHITECTURE
    }

    results = run_full_pate_experiment(main_cfg)
    plot_privacy_loss_composition(results)
    print("Workflow completed successfully!")

if __name__ == "__main__":
    main()

Mounted at /content/gdrive
Starting consolidated execution block...

RUNNING PATE ACCOUNTING COMPLIANCE TESTS
PASS: Direct multiclass q verification
PASS: Independent hand-calculated DI regression test
All core compliance unit tests passed successfully!

Executing experiment run: PATE: 9 Teachers (3 LSTM, 3 GRU, 3 CNN), 1 MLP Student
Data-Dependent Epsilon: 84.509852
Data-Independent Epsilon: 267691.159779
Workflow completed successfully!


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import random
import math

# =====================================================================
# DEFINE MISSING STUDENT ARCHITECTURES WITH PROFESSIONAL DOCUMENTATION
# =====================================================================

class StudentLSTM(nn.Module):
    """
    Recurrent Neural Network Student model utilizing an LSTM layer to map features
    over sequential or pseudo-sequential tabular steps.
    """
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        # Core LSTM unit processing input sequences
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        # Fully connected layer maps hidden dimensions to output logit dimensions
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Convert 2D tabular tensor [Batch, Features] to pseudo-sequence 3D tensor [Batch, Seq=1, Features]
        if x.dim() == 2:
            x = x.unsqueeze(1)
        # Propagate sequence through LSTM layers
        _, (hidden, _) = self.lstm(x)
        # Extract final layer hidden state representation and calculate log-softmax
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentGRU(nn.Module):
    """
    Recurrent Neural Network Student model utilizing a Gated Recurrent Unit (GRU)
    for computational efficiency compared to LSTMs.
    """
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        # Gated Recurrent Unit processing block
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        # Classification projection head mapping to class labels
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Convert 2D tabular input into sequence-like 3D input tensor
        if x.dim() == 2:
            x = x.unsqueeze(1)
        # Forward pass through GRU blocks
        _, hidden = self.gru(x)
        # project final hidden step to log-probabilities
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentCNN(nn.Module):
    """
    One-Dimensional Convolutional Neural Network (1D-CNN) Student model designed
    to extract localized patterns across tabular feature arrays.
    """
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        # Convolutional layer 1: local receptive field feature extractor
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        # Convolutional layer 2: deeper high-level representations
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        # Linear transition layer from spatial feature maps
        self.fc1 = nn.Linear(64 * input_size, 128)
        # Final linear mapping layer for prediction logits
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Reshape input tensor from [Batch, Features] to 1D convolution compatible shape [Batch, Channels=1, Width=Features]
        if x.dim() == 2:
            x = x.unsqueeze(1)
        # Convolve, apply Rectified Linear Unit activations
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # Flatten spatial structures into a 1D tensor vector per sample
        x = torch.flatten(x, start_dim=1)
        # Output projection head to map to classification target log-space
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

def build_student(architecture, input_size, num_classes):
    """
    Factory function to instantiate student models dynamically based on input configurations.
    """
    if architecture == "MLP":
        return StudentMLP(input_size=input_size, num_classes=num_classes)
    elif architecture == "LSTM":
        return StudentLSTM(input_size=input_size, num_classes=num_classes)
    elif architecture == "GRU":
        return StudentGRU(input_size=input_size, num_classes=num_classes)
    elif architecture == "CNN":
        return StudentCNN(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown student architecture: {architecture}")

# =====================================================================
# RIGOROUS EVALUATION FUNCTION (AUC, MAE, ACCURACY, PRECISION, RECALL, F1)
# =====================================================================
def evaluate_student_rigorous(student_model, X_test, y_test, num_classes, batch_size, device):
    """
    Performs multi-metric performance evaluation on unseen testing splits to
    compute accuracy, macro precision, recall, F1-scores, AUC, and multi-class MAE.
    """
    student_model.eval() # Configure model to evaluation (inference) state
    loader = DataLoader(TabularDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    all_predictions = []
    all_probs = []

    # Disable PyTorch automatic gradient computation to optimize memory usage
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = student_model(inputs) # Get output log probabilities
            probs = torch.exp(log_probabilities) # Convert log probabilities back to original distribution space
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            all_predictions.extend(batch_predictions.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_probs = np.array(all_probs, dtype=np.float64)
    y_test = np.array(y_test, dtype=np.float64)

    # Standard multi-class performance statistics metrics calculations
    accuracy = accuracy_score(y_test, all_predictions)
    precision = precision_score(y_test, all_predictions, average='macro', zero_division=0)
    recall = recall_score(y_test, all_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, all_predictions, average='macro', zero_division=0)

    # Explicitly map binary vs multiclass shapes to calculate valid parameters
    if num_classes == 2:
        # For 2 classes, extract the positive class probabilities to form a (N,) shape
        target_probs = all_probs[:, 1]
        mae = float(np.mean(np.abs(y_test - target_probs)))
        try:
            auc = float(roc_auc_score(y_test, target_probs))
        except Exception:
            auc = np.nan
    else:
        # For 3+ classes, binarized labels are a 2D matrix of shape [Samples, Classes]
        y_test_bin = label_binarize(y_test, classes=np.arange(num_classes)).astype(np.float64)
        mae = float(mean_absolute_error(y_test_bin, all_probs))
        try:
            auc = float(roc_auc_score(y_test_bin, all_probs, average='macro', multi_class='ovr'))
        except Exception:
            auc = np.nan

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1_macro),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "mae": float(mae)
    }

# =====================================================================
# SYSTEMATIC CONFIGURATION EXPERIMENT PIPELINE
# =====================================================================
def run_custom_pate_experiment(config):
    """
    Orchestrates the entire PATE lifecycle for a single scenario: loading,
    partitioning, scaling, ensemble training, Laplace sanitization, private accounting,
    and student distillations. Every run uses the full dataset independently.
    """
    # Set global deterministic random number generator states
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # 1. Pipeline Segment: IO and Preprocessing (loads the full dataset fresh)
    data = load_dataset(DATA_PATH, TARGET_COLUMN, SELECTED_CLASSES)
    X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder = preprocess_and_split(data, TARGET_COLUMN, seed=SEED)

    input_size = X_teacher.shape[1]
    num_classes = len(np.unique(y_teacher))

    # Fit standard scaler to teacher features and transform auxiliary divisions
    scaler = StandardScaler()
    X_teacher_scaled = scaler.fit_transform(X_teacher)
    X_public_scaled = scaler.transform(X_public)
    X_final_test_scaled = scaler.transform(X_final_test)

    # 2. Pipeline Segment: Splitting Teacher Shards
    n = len(X_teacher_scaled)
    indices = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    partitions = np.array_split(indices, config["num_teachers"])

    # Generate disjoint loaders to train parallel individual models
    train_loaders, valid_loaders = [], []
    for teacher_id, teacher_indices in enumerate(partitions):
        X_part = X_teacher_scaled[teacher_indices]
        y_part = y_teacher[teacher_indices]
        X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + teacher_id, stratify=y_part)
        if USE_OVERSAMPLING:
            # Apply oversampling on minor classes to prevent learning bias inside localized partitions
            sampler = RandomOverSampler(random_state=SEED + teacher_id)
            X_train, y_train = sampler.fit_resample(X_train, y_train)
        train_loaders.append(DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True))
        valid_loaders.append(DataLoader(TabularDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False))

    # 3. Pipeline Segment: Train Parallel Ensemble
    teachers = []
    for idx in range(config["num_teachers"]):
        arch = config["teacher_archs"][idx % len(config["teacher_archs"])]
        model = build_teacher(arch, input_size, num_classes)
        trained_model, _ = train_model(model, train_loaders[idx], valid_loaders[idx], TEACHER_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)
        teachers.append(trained_model)

    # 4. Pipeline Segment: Aggregation & Privacy Calculations
    public_loader = DataLoader(TabularDataset(X_public_scaled, y_public), batch_size=BATCH_SIZE, shuffle=False)
    all_teacher_preds = []
    for idx, t_model in enumerate(teachers):
        preds = predict_labels(t_model, public_loader, DEVICE)
        all_teacher_preds.append(preds)
    all_teacher_preds = np.stack(all_teacher_preds)

    # Secure aggregated predictions using Laplace noise mechanism
    released_labels, vote_counts, _ = aggregate_teacher_votes_laplace(all_teacher_preds, num_classes, LAPLACE_INVERSE_SCALE, random_seed=SEED)
    privacy_results = calculate_pate_privacy(vote_counts, LAPLACE_INVERSE_SCALE, DELTA, MAX_MOMENT)

    mean_vote_value = float(np.mean(vote_counts))

    # 5. Pipeline Segment: Train Private Student Distillation Model
    X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED, stratify=released_labels)
    student_train_loader = DataLoader(TabularDataset(X_student_train, y_student_train), batch_size=BATCH_SIZE, shuffle=True)
    student_valid_loader = DataLoader(TabularDataset(X_student_valid, y_student_valid), batch_size=BATCH_SIZE, shuffle=False)

    student_model = build_student(config["student_arch"], input_size, num_classes)
    trained_student, _ = train_model(student_model, student_train_loader, student_valid_loader, STUDENT_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)

    # Run extensive, bias-free metrics evaluation on the student
    student_metrics = evaluate_student_rigorous(trained_student, X_final_test_scaled, y_final_test, num_classes, BATCH_SIZE, DEVICE)

    return {
        "dep_eps": privacy_results["data_dependent_epsilon"],
        "ind_eps": privacy_results["data_independent_epsilon"],
        "mean_vote": mean_vote_value,
        **student_metrics
    }

# =====================================================================
# DEFINE THE 10 ARCHITECTURAL CONFIGURATIONS
# =====================================================================
configurations = [
    {
        "name": "Config 1 (2 LSTM, 2 GRU, 2 CNN -> 1 MLP Student)",
        "num_teachers": 6,
        "teacher_archs": ["LSTM", "LSTM", "GRU", "GRU", "CNN", "CNN"],
        "student_arch": "MLP"
    },
    {
        "name": "Config 2 (1 CNN Teacher -> 1 CNN Student)",
        "num_teachers": 1,
        "teacher_archs": ["CNN"],
        "student_arch": "CNN"
    },
    {
        "name": "Config 3 (1 LSTM Teacher -> 1 LSTM Student)",
        "num_teachers": 1,
        "teacher_archs": ["LSTM"],
        "student_arch": "LSTM"
    },
    {
        "name": "Config 4 (1 MLP Teacher -> 1 MLP Student)",
        "num_teachers": 1,
        "teacher_archs": ["MLP"],
        "student_arch": "MLP"
    },
    {
        "name": "Config 5 (6 CNN, 6 GRU, 6 LSTM -> 1 CNN Student)",
        "num_teachers": 18,
        "teacher_archs": ["CNN"]*6 + ["GRU"]*6 + ["LSTM"]*6,
        "student_arch": "CNN"
    },
    {
        "name": "Config 6 (4 MLP, 4 CNN, 4 GRU, 4 LSTM -> 1 CNN Student)",
        "num_teachers": 16,
        "teacher_archs": ["MLP"]*4 + ["CNN"]*4 + ["GRU"]*4 + ["LSTM"]*4,
        "student_arch": "CNN"
    },
    {
        "name": "Config 7 (3 MLP Teachers -> 1 MLP Student)",
        "num_teachers": 3,
        "teacher_archs": ["MLP"]*3,
        "student_arch": "MLP"
    },
    {
        "name": "Config 8 (3 LSTM, 3 GRU, 3 CNN -> 1 MLP Student)",
        "num_teachers": 9,
        "teacher_archs": ["LSTM"]*3 + ["GRU"]*3 + ["CNN"]*3,
        "student_arch": "MLP"
    },
    {
        "name": "Config 9 (3 CNN Teachers -> 1 CNN Student)",
        "num_teachers": 3,
        "teacher_archs": ["CNN"]*3,
        "student_arch": "CNN"
    },
    {
        "name": "Config 10 (3 LSTM, 4 MLP, 3 CNN -> 1 GRU Student)",
        "num_teachers": 10,
        "teacher_archs": ["LSTM"]*3 + ["MLP"]*4 + ["CNN"]*3,
        "student_arch": "GRU"
    }
]

# =====================================================================
# RUN SYSTEMATIC ENSEMBLE CONFIGURATION COMPILATION
# =====================================================================
results_table = []
for idx, config in enumerate(configurations, 1):
    print(f"Executing scenario {idx}/10: {config['name']}...")
    try:
        metrics = run_custom_pate_experiment(config)
        metrics["Configuration"] = config["name"]
        results_table.append(metrics)
    except Exception as e:
        print(f"Failed scenario {config['name']}: {str(e)}")

# Display summary comparisons in structured Pandas DataFrame form
df_results = pd.DataFrame(results_table)
cols_order = ["Configuration", "dep_eps", "ind_eps", "accuracy", "precision", "recall", "f1_score", "auc", "mae", "mean_vote"]
df_results = df_results[[c for c in cols_order if c in df_results.columns]]
display(df_results)

Executing scenario 1/10: Config 1 (2 LSTM, 2 GRU, 2 CNN -> 1 MLP Student)...
Executing scenario 2/10: Config 2 (1 CNN Teacher -> 1 CNN Student)...
Executing scenario 3/10: Config 3 (1 LSTM Teacher -> 1 LSTM Student)...
Executing scenario 4/10: Config 4 (1 MLP Teacher -> 1 MLP Student)...
Executing scenario 5/10: Config 5 (6 CNN, 6 GRU, 6 LSTM -> 1 CNN Student)...
Executing scenario 6/10: Config 6 (4 MLP, 4 CNN, 4 GRU, 4 LSTM -> 1 CNN Student)...
Executing scenario 7/10: Config 7 (3 MLP Teachers -> 1 MLP Student)...
Executing scenario 8/10: Config 8 (3 LSTM, 3 GRU, 3 CNN -> 1 MLP Student)...
Executing scenario 9/10: Config 9 (3 CNN Teachers -> 1 CNN Student)...
Executing scenario 10/10: Config 10 (3 LSTM, 4 MLP, 3 CNN -> 1 GRU Student)...


,Configuration,dep_eps,ind_eps,accuracy,precision,recall,f1_score,auc,mae,mean_vote
0,"Config 1 (2 LSTM, 2 GRU, 2 CNN -> 1 MLP Student)",2572.109945,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999977,0.000645,3.0
1,Config 2 (1 CNN Teacher -> 1 CNN Student),267691.159779,267691.159779,0.999751,0.999755,0.999746,0.999750,1.000000,0.237222,0.5
2,Config 3 (1 LSTM Teacher -> 1 LSTM Student),267691.159779,267691.159779,0.999362,0.999354,0.999370,0.999362,0.999847,0.239235,0.5
3,Config 4 (1 MLP Teacher -> 1 MLP Student),267691.159779,267691.159779,0.999479,0.999473,0.999483,0.999478,0.999992,0.232238,0.5
4,"Config 5 (6 CNN, 6 GRU, 6 LSTM -> 1 CNN Student)",3.686763,267691.159779,0.999951,0.999950,0.999952,0.999951,0.999946,0.000049,9.0
5,"Config 6 (4 MLP, 4 CNN, 4 GRU, 4 LSTM -> 1 CNN...",4.293510,267691.159779,0.999951,0.999950,0.999952,0.999951,0.999920,0.000049,8.0
6,Config 7 (3 MLP Teachers -> 1 MLP Student),70124.172144,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999997,0.028715,1.5
7,"Config 8 (3 LSTM, 3 GRU, 3 CNN -> 1 MLP Student)",84.509852,267691.159779,0.999958,0.999957,0.999959,0.999958,0.999964,0.000054,4.5
8,Config 9 (3 CNN Teachers -> 1 CNN Student),70131.848211,267691.159779,0.999945,0.999944,0.999946,0.999945,0.999975,0.027872,1.5
9,"Config 10 (3 LSTM, 4 MLP, 3 CNN -> 1 GRU Student)",33.274447,267691.159779,0.999955,0.999953,0.999956,0.999955,0.999986,0.000086,5.0


In [ ]:
import os
import sys
import copy
import math
import random
import time
import warnings
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, roc_auc_score
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import RandomOverSampler

# Suppress warning outputs to maintain clean terminal execution logs
warnings.filterwarnings('ignore')

# =====================================================================
# DYNAMIC PATH RESOLUTION WITH COLA DRIVE SUPPORT
# =====================================================================
def find_data_file(filename="data.csv"):
    search_paths = [
        "/content/gdrive/MyDrive",
        "/content/gdrive/My Drive",
        "/content/drive/MyDrive",
        "/content/drive/My Drive",
        "/content/gdrive",
        "/content/drive",
        "/content"
    ]
    for path in search_paths:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            return full_path
    # Exhaustive search fallback
    for root, dirs, files in os.walk("/content"):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not locate '{filename}' in the notebook directory hierarchy.")

try:
    DATA_PATH = find_data_file("data.csv")
    print(f"Resolved dataset location to: {DATA_PATH}")
except Exception as e:
    print(f"Warning: {e}. Defaulting to standard fallback path.")
    DATA_PATH = "/content/gdrive/MyDrive/data.csv"

# =====================================================================
# SELF-CONTAINED GLOBAL HYPERPARAMETERS
# =====================================================================
SEED = 42
TARGET_COLUMN = "is_attack"
SELECTED_CLASSES = [0, 1]
BATCH_SIZE = 128
TEACHER_EPOCHS = 23
STUDENT_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE = 3
TEACHER_VALID_FRACTION = 0.30
STUDENT_VALID_FRACTION = 0.20
TEACHER_POOL_FRACTION = 0.80
STUDENT_PRIVATE_FRACTION = 0.00
PUBLIC_FRACTION = 0.05
FINAL_TEST_FRACTION = 0.15

USE_OVERSAMPLING = True
LAPLACE_INVERSE_SCALE = 1.3
DELTA = 1e-5
MAX_MOMENT = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# PIPELINE FUNCTIONS & DATA UTILITIES
# =====================================================================
def load_dataset(path, target_column, selected_classes):
    data = pd.read_csv(path, index_col=0)
    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' was not found.")
    data = data[data[target_column].isin(selected_classes)].copy()
    if len(data) == 0:
        raise ValueError("No observations remain after class filtering.")
    return data

def preprocess_and_split(data, target_column, seed=42):
    data = data.copy()
    total_fraction = TEACHER_POOL_FRACTION + STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    if not np.isclose(total_fraction, 1.0):
        raise ValueError("Dataset fractions must sum to 1.0.")

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(data[target_column])
    X = data.drop(columns=[target_column])

    non_numeric = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(non_numeric) > 0:
        X = X.drop(columns=non_numeric)

    X = X.apply(pd.to_numeric, errors="coerce")
    valid_rows = ~X.isna().any(axis=1)
    X = X.loc[valid_rows].reset_index(drop=True)
    y = y[valid_rows.to_numpy()]

    X = X.to_numpy(dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    X_teacher, X_heldout, y_teacher, y_heldout = train_test_split(
        X, y, test_size=(1.0 - TEACHER_POOL_FRACTION), random_state=seed, stratify=y
    )

    heldout_total_sub_fraction = STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    test_size_final_test = FINAL_TEST_FRACTION / heldout_total_sub_fraction

    X_public, X_final_test, y_public, y_final_test = train_test_split(
        X_heldout, y_heldout, test_size=test_size_final_test, random_state=seed, stratify=y_heldout
    )

    X_private = np.zeros((0, X_teacher.shape[1]), dtype=np.float32)
    y_private = np.zeros((0,), dtype=np.int64)

    return X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

# =====================================================================
# TEACHER ARCHITECTURES
# =====================================================================
class TeacherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherCNN(nn.Module):
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class TeacherMLP(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

def build_teacher(architecture, input_size, num_classes):
    if architecture == "LSTM":
         return TeacherLSTM(input_size=input_size, num_classes=num_classes)
    if architecture == "GRU":
         return TeacherGRU(input_size=input_size, num_classes=num_classes)
    if architecture == "CNN":
        return TeacherCNN(input_size=input_size, num_classes=num_classes)
    if architecture == "MLP":
        return TeacherMLP(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown teacher architecture: {architecture}")

# =====================================================================
# STUDENT ARCHITECTURES
# =====================================================================
class StudentLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentCNN(nn.Module):
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class StudentMLP(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

def build_student(architecture, input_size, num_classes):
    if architecture == "MLP":
        return StudentMLP(input_size=input_size, num_classes=num_classes)
    elif architecture == "LSTM":
        return StudentLSTM(input_size=input_size, num_classes=num_classes)
    elif architecture == "GRU":
        return StudentGRU(input_size=input_size, num_classes=num_classes)
    elif architecture == "CNN":
        return StudentCNN(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown student architecture: {architecture}")

# =====================================================================
# UTILITIES & RECURRENT TRAINING SCHEDULERS
# =====================================================================
def train_model(model, train_loader, valid_loader, epochs, learning_rate, patience, device):
    model = model.to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_validation_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        total_train_loss, total_train, correct_train = 0.0, 0, 0
        for is_input, labels in train_loader:
            is_input, labels = is_input.to(device), labels.to(device)
            optimizer.zero_grad()
            log_probabilities = model(is_input)
            loss = criterion(log_probabilities, labels)
            loss.backward()
            optimizer.step()
            b_size = labels.size(0)
            total_train_loss += loss.item() * b_size
            predictions = torch.argmax(log_probabilities, dim=1)
            correct_train += (predictions == labels).sum().item()
            total_train += b_size

        model.eval()
        total_valid_loss, total_valid, correct_valid = 0.0, 0, 0
        with torch.no_grad():
            for is_input, labels in valid_loader:
                is_input, labels = is_input.to(device), labels.to(device)
                log_probabilities = model(is_input)
                loss = criterion(log_probabilities, labels)
                b_size = labels.size(0)
                total_valid_loss += loss.item() * b_size
                predictions = torch.argmax(log_probabilities, dim=1)
                correct_valid += (predictions == labels).sum().item()
                total_valid += b_size

        valid_loss = total_valid_loss / total_valid
        history.append({"epoch": epoch + 1, "valid_loss": valid_loss})

        if valid_loss < best_validation_loss:
            best_validation_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

def predict_labels(model, loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = model(inputs)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            predictions.extend(batch_predictions.cpu().numpy())
    return np.asarray(predictions, dtype=np.int64)

# =====================================================================
# PRIVACY AGGREGATION & ACCOUNTING
# =====================================================================
def teacher_predictions_to_counts(teacher_preds, num_classes):
    teacher_preds = np.asarray(teacher_preds, dtype=np.int64)
    num_teachers, num_queries = teacher_preds.shape
    counts = np.zeros((num_queries, num_classes), dtype=np.int64)
    for query_index in range(num_queries):
        counts[query_index] = np.bincount(teacher_preds[:, query_index], minlength=num_classes)
    return counts

def aggregate_teacher_votes_laplace(teacher_preds, num_classes, laplace_inverse_scale, random_seed=None):
    counts = teacher_predictions_to_counts(teacher_preds, num_classes)
    rng = np.random.default_rng(random_seed)
    laplace_scale = 1.0 / laplace_inverse_scale
    noise = rng.laplace(loc=0.0, scale=laplace_scale, size=counts.shape)
    noisy_counts = counts.astype(np.float64) + noise
    released_labels = np.argmax(noisy_counts, axis=1)
    return released_labels, counts, noisy_counts

def logaddexp_scalar(a, b):
    if a == -math.inf: return b
    if b == -math.inf: return a
    maximum = max(a, b)
    return maximum + math.log(math.exp(a - maximum) + math.exp(b - maximum))

def logmgf_exact(q, mechanism_eps, moment_order):
    lam = float(moment_order)
    eps = float(mechanism_eps)
    bound_1 = 0.5 * (eps ** 2) * lam * (lam + 1.0)
    bound_3 = eps * lam
    bound_2 = math.inf
    denominator = 1.0 - math.exp(eps) * q
    if q < 0.5 and denominator > 0.0:
        log_one_minus_q = math.log1p(-q)
        log_term_1 = log_one_minus_q + lam * (log_one_minus_q - math.log(denominator))
        log_term_2 = math.log(q) + eps * lam if q > 0.0 else -math.inf
        bound_2 = logaddexp_scalar(log_term_1, log_term_2)
    return min(bound_1, bound_2, bound_3)

def compute_q_noisy_max(counts, laplace_inverse_scale):
    counts = np.asarray(counts, dtype=np.float64)
    num_classes = len(counts)
    if num_classes < 2:
        return 0.0
    winner = int(np.argmax(counts))
    winning_votes = counts[winner]
    competitors = np.delete(counts, winner)
    gaps = laplace_inverse_scale * (winning_votes - competitors)
    q = float(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0))
    return min(max(q, 0.0), 1.0 - 1.0 / num_classes)

def data_dependent_log_moment(counts, laplace_inverse_scale, moment_order):
    q = compute_q_noisy_max(counts, laplace_inverse_scale)
    mechanism_eps = 2.0 * laplace_inverse_scale
    return logmgf_exact(q, mechanism_eps, moment_order)

def data_independent_log_moment(laplace_inverse_scale, moment_order):
    mechanism_eps = 2.0 * laplace_inverse_scale
    lam = float(moment_order)
    concentration_bound = 0.5 * (mechanism_eps ** 2) * lam * (lam + 1.0)
    direct_dp_bound = mechanism_eps * lam
    return min(concentration_bound, direct_dp_bound)

def calculate_pate_privacy(vote_counts, laplace_inverse_scale=1.3, delta=1e-5, moments=32):
    vote_counts = np.asarray(vote_counts, dtype=np.int64)
    num_queries, classes_dim = vote_counts.shape
    orders = np.arange(1, moments + 1, dtype=np.float64)
    dependent_total = np.zeros(moments, dtype=np.float64)
    independent_total = np.zeros(moments, dtype=np.float64)

    for counts in vote_counts:
        for pos, order in enumerate(orders):
            dependent_total[pos] += data_dependent_log_moment(counts, laplace_inverse_scale, order)
            independent_total[pos] += data_independent_log_moment(laplace_inverse_scale, order)

    dep_candidates = (dependent_total - math.log(delta)) / orders
    ind_candidates = (independent_total - math.log(delta)) / orders
    dep_idx = np.argmin(dep_candidates)
    ind_idx = np.argmin(ind_candidates)

    return {
        "data_dependent_epsilon": float(dep_candidates[dep_idx]),
        "data_independent_epsilon": float(ind_candidates[ind_idx])
    }

# =====================================================================
# RIGOROUS STUDENT EVALUATION
# =====================================================================
def evaluate_student_rigorous(student_model, X_test, y_test, num_classes, batch_size, device):
    student_model.eval()
    loader = DataLoader(TabularDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    all_predictions = []
    all_probs = []

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = student_model(inputs)
            probs = torch.exp(log_probabilities)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            all_predictions.extend(batch_predictions.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_probs = np.array(all_probs, dtype=np.float64)
    y_test = np.array(y_test, dtype=np.float64)

    accuracy = accuracy_score(y_test, all_predictions)
    precision = precision_score(y_test, all_predictions, average='macro', zero_division=0)
    recall = recall_score(y_test, all_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, all_predictions, average='macro', zero_division=0)

    if num_classes == 2:
        target_probs = all_probs[:, 1]
        mae = float(np.mean(np.abs(y_test - target_probs)))
        try:
            auc = float(roc_auc_score(y_test, target_probs))
        except Exception:
            auc = np.nan
    else:
        y_test_bin = label_binarize(y_test, classes=np.arange(num_classes)).astype(np.float64)
        mae = float(mean_absolute_error(y_test_bin, all_probs))
        try:
            auc = float(roc_auc_score(y_test_bin, all_probs, average='macro', multi_class='ovr'))
        except Exception:
            auc = np.nan

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1_macro),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "mae": float(mae)
    }

# =====================================================================
# EXPERIMENT RUNNER
# =====================================================================
def run_custom_pate_experiment(config):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    data = load_dataset(DATA_PATH, TARGET_COLUMN, SELECTED_CLASSES)
    X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder = preprocess_and_split(data, TARGET_COLUMN, seed=SEED)

    input_size = X_teacher.shape[1]
    num_classes = len(np.unique(y_teacher))

    scaler = StandardScaler()
    X_teacher_scaled = scaler.fit_transform(X_teacher)
    X_public_scaled = scaler.transform(X_public)
    X_final_test_scaled = scaler.transform(X_final_test)

    n = len(X_teacher_scaled)
    indices = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    partitions = np.array_split(indices, config["num_teachers"])

    train_loaders, valid_loaders = [], []
    for teacher_id, teacher_indices in enumerate(partitions):
        X_part = X_teacher_scaled[teacher_indices]
        y_part = y_teacher[teacher_indices]
        X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + teacher_id, stratify=y_part)
        if USE_OVERSAMPLING:
            sampler = RandomOverSampler(random_state=SEED + teacher_id)
            X_train, y_train = sampler.fit_resample(X_train, y_train)
        train_loaders.append(DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True))
        valid_loaders.append(DataLoader(TabularDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False))

    teachers = []
    for idx in range(config["num_teachers"]):
        arch = config["teacher_archs"][idx % len(config["teacher_archs"])]
        model = build_teacher(arch, input_size, num_classes)
        trained_model, _ = train_model(model, train_loaders[idx], valid_loaders[idx], TEACHER_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)
        teachers.append(trained_model)

    public_loader = DataLoader(TabularDataset(X_public_scaled, y_public), batch_size=BATCH_SIZE, shuffle=False)
    all_teacher_preds = []
    for idx, t_model in enumerate(teachers):
        preds = predict_labels(t_model, public_loader, DEVICE)
        all_teacher_preds.append(preds)
    all_teacher_preds = np.stack(all_teacher_preds)

    released_labels, vote_counts, _ = aggregate_teacher_votes_laplace(all_teacher_preds, num_classes, LAPLACE_INVERSE_SCALE, random_seed=SEED)
    privacy_results = calculate_pate_privacy(vote_counts, LAPLACE_INVERSE_SCALE, DELTA, MAX_MOMENT)

    mean_vote_value = float(np.mean(vote_counts))

    X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED, stratify=released_labels)
    student_train_loader = DataLoader(TabularDataset(X_student_train, y_student_train), batch_size=BATCH_SIZE, shuffle=True)
    student_valid_loader = DataLoader(TabularDataset(X_student_valid, y_student_valid), batch_size=BATCH_SIZE, shuffle=False)

    student_model = build_student(config["student_arch"], input_size, num_classes)
    trained_student, _ = train_model(student_model, student_train_loader, student_valid_loader, STUDENT_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)

    student_metrics = evaluate_student_rigorous(trained_student, X_final_test_scaled, y_final_test, num_classes, BATCH_SIZE, DEVICE)

    return {
        "dep_eps": privacy_results["data_dependent_epsilon"],
        "ind_eps": privacy_results["data_independent_epsilon"],
        "mean_vote": mean_vote_value,
        **student_metrics
    }

# =====================================================================
# DEFINE THE 5 NEW CONFIGURATIONS FOR THE EXPERIMENT
# =====================================================================
new_configurations = [
    {
        "name": "Config 11 (10 CNN Teachers -> 1 CNN Student)",
        "num_teachers": 10,
        "teacher_archs": ["CNN"] * 10,
        "student_arch": "CNN"
    },
    {
        "name": "Config 12 (3 LSTM Teachers -> 1 LSTM Student)",
        "num_teachers": 3,
        "teacher_archs": ["LSTM"] * 3,
        "student_arch": "LSTM"
    },
    {
        "name": "Config 13 (1 LSTM, 1 CNN, 1 MLP Teachers -> 1 LSTM Student)",
        "num_teachers": 3,
        "teacher_archs": ["LSTM", "CNN", "MLP"],
        "student_arch": "LSTM"
    },
    {
        "name": "Config 14 (1 LSTM, 1 MLP, 1 CNN Teachers -> 1 GRU Student)",
        "num_teachers": 3,
        "teacher_archs": ["LSTM", "MLP", "CNN"],
        "student_arch": "GRU"
    },
    {
        "name": "Config 15 (1 LSTM Teacher -> 1 GRU Student)",
        "num_teachers": 1,
        "teacher_archs": ["LSTM"],
        "student_arch": "GRU"
    }
]

# =====================================================================
# RUN SYSTEMATIC ENSEMBLE CONFIGURATION COMPILATION
# =====================================================================
new_results_table = []
for idx, config in enumerate(new_configurations, 11):
    print(f"Executing scenario {idx}/15: {config['name']}...")
    try:
        metrics = run_custom_pate_experiment(config)
        metrics["Configuration"] = config["name"]
        new_results_table.append(metrics)
    except Exception as e:
        print(f"Failed scenario {config['name']}: {str(e)}")

# Display summary comparisons in structured Pandas DataFrame form
df_new_results = pd.DataFrame(new_results_table)
cols_order = ["Configuration", "dep_eps", "ind_eps", "accuracy", "precision", "recall", "f1_score", "auc", "mae", "mean_vote"]
df_new_results = df_new_results[[c for c in cols_order if c in df_new_results.columns]]
display(df_new_results)


Resolved dataset location to: /content/gdrive/MyDrive/data.csv
Executing scenario 11/15: Config 11 (10 CNN Teachers -> 1 CNN Student)...
Executing scenario 12/15: Config 12 (3 LSTM Teachers -> 1 LSTM Student)...
Executing scenario 13/15: Config 13 (1 LSTM, 1 CNN, 1 MLP Teachers -> 1 LSTM Student)...
Executing scenario 14/15: Config 14 (1 LSTM, 1 MLP, 1 CNN Teachers -> 1 GRU Student)...
Executing scenario 15/15: Config 15 (1 LSTM Teacher -> 1 GRU Student)...


,Configuration,dep_eps,ind_eps,accuracy,precision,recall,f1_score,auc,mae,mean_vote
0,Config 11 (10 CNN Teachers -> 1 CNN Student),33.305523,267691.159779,0.999951,0.999950,0.999952,0.999951,0.999937,0.000049,5.0
1,Config 12 (3 LSTM Teachers -> 1 LSTM Student),70124.172144,267691.159779,0.999780,0.999783,0.999776,0.999780,0.999956,0.037627,1.5
2,"Config 13 (1 LSTM, 1 CNN, 1 MLP Teachers -> 1 ...",70124.172144,267691.159779,0.999780,0.999783,0.999776,0.999780,0.999940,0.036584,1.5
3,"Config 14 (1 LSTM, 1 MLP, 1 CNN Teachers -> 1 ...",70124.172144,267691.159779,0.999780,0.999783,0.999776,0.999780,0.999933,0.036885,1.5
4,Config 15 (1 LSTM Teacher -> 1 GRU Student),267691.159779,267691.159779,0.997844,0.997909,0.997783,0.997842,0.999855,0.249215,0.5
